# Build an Agent Swarm with CrewAI and Gemini 3.5 Flash

Build a four-agent workflow that researches, writes, reviews, and finalizes a short article.


## 1. What we are building

| Agent | Job |
| --- | --- |
| Researcher | Finds current information |
| Writer | Creates the draft |
| Reviewer | Improves clarity and accuracy |
| Manager | Delegates work, validates results, and coordinates the final response |

CrewAI runs the workers as a hierarchical swarm managed by the Content Coordinator.


## 2. Install dependencies

Run the next cell once to install CrewAI, Gemini support, Olostep, and the Jupyter event-loop helper.


In [1]:
# %pip install --quiet -U \
#     crewai \
#     crewai-tools \
#     google-genai \
#     litellm \
#     olostep \
#     nest-asyncio

# Print installed versions so we can verify the install worked.
import importlib.metadata as md

for pkg in [
    "crewai",
    "crewai-tools",
    "google-genai",
    "litellm",
    "olostep",
    "nest-asyncio",
]:
    print(f"{pkg:>16}  {md.version(pkg)}")


          crewai  1.14.6
    crewai-tools  1.14.6
    google-genai  2.8.0
         litellm  1.87.1
         olostep  1.1.0
    nest-asyncio  1.6.0


## 3. Set environment variables

Create your API keys, then set them before running the notebook.

**PowerShell**

```powershell
$env:GEMINI_API_KEY="your-gemini-key"
$env:OLOSTEP_API_KEY="your-olostep-key"
```

**macOS/Linux**

```bash
export GEMINI_API_KEY="your-gemini-key"
export OLOSTEP_API_KEY="your-olostep-key"
```

Get keys from [Google AI Studio](https://aistudio.google.com/app/apikey) and the [Olostep dashboard](https://www.olostep.com/dashboard/). Restart the kernel after changing variables.


In [2]:
import os

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise RuntimeError(
        "Set GEMINI_API_KEY or GOOGLE_API_KEY in your environment, "
        "then restart the notebook kernel."
    )

masked = api_key[:4] + "..." + api_key[-4:]
print(f"Gemini API key loaded")


Gemini API key loaded


## 4. Configure Gemini

Create one shared CrewAI `LLM` and reuse it for all agents.


In [3]:
from crewai import LLM


GEMINI_MODEL = "gemini/gemini-3.5-flash"


gemini_llm = LLM(
    model=GEMINI_MODEL,
    api_key=os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY"),
    temperature=1.0,  # Google recommends 1.0 for the Gemini 3 family.
)


print(f"LLM ready: {gemini_llm.model}")

print("Provider API: Google Gemini (via LiteLLM)")


LLM ready: gemini-3.5-flash
Provider API: Google Gemini (via LiteLLM)


## 5. Add live web access

The Researcher uses one Olostep tool. On every call, the agent decides whether to return search summaries only or also scrape the result pages.


> If Olostep is missing, run `%pip install -U olostep`.


In [4]:
from pydantic import Field, PrivateAttr
from crewai.tools import BaseTool

try:
    from olostep import Olostep, Olostep_BaseError
except ImportError:
    raise ImportError(
        "The 'olostep' package is not installed. Run `%pip install -U olostep`."
    )


olostep_api_key = os.getenv("OLOSTEP_API_KEY")

if not olostep_api_key or olostep_api_key == "your-olostep-api-key-here":
    print(
        "The Olostep tool will fail at call time unless OLOSTEP_API_KEY "
        "is set in the environment."
    )
else:
    print("Olostep API key loaded.")


_olostep_client = Olostep(api_key=olostep_api_key) if olostep_api_key else None


class OlostepSearchTool(BaseTool):
    """Search the web with a strict research-call budget."""

    name: str = "olostep_web_search"
    description: str = (
        "Search the live web. Set scrape=false for fast discovery and "
        "scrape=true for full-page evidence. You have at most 3 discovery "
        "searches without scraping and 6 external search calls total. After "
        "3 discovery searches, use scrape=true on the strongest query or "
        "domain. Each call returns at most 3 results. Inputs: query, scrape, "
        "limit, and optional include_domains."
    )

    default_limit: int = Field(default=3, ge=1, le=3)
    max_chars_per_page: int = Field(default=4000, ge=500, le=12000)

    _total_calls: int = PrivateAttr(default=0)
    _discovery_calls: int = PrivateAttr(default=0)
    _call_history: list[dict] = PrivateAttr(default_factory=list)

    def _run(
        self,
        query: str,
        scrape: bool = False,
        limit: int | None = None,
        include_domains: list[str] | None = None,
    ) -> str:
        if _olostep_client is None:
            return (
                "OLOSTEP_API_KEY is not set. Set it in the environment "
                "and restart the kernel."
            )

        if self._total_calls >= 6:
            return (
                "Research search budget exhausted: 6 external calls have "
                "already been used. Stop searching and complete the task "
                "with the collected sources."
            )

        if not scrape and self._discovery_calls >= 3:
            return (
                "Discovery-search limit reached. Do not run another "
                "scrape=False search. Use scrape=True on the strongest query "
                "or selected domains, or finish with the sources already found."
            )

        clean_query = query.strip()
        if not clean_query:
            return "Search query cannot be empty."

        result_limit = max(1, min(limit or self.default_limit, 3))
        kwargs = {"query": clean_query, "limit": result_limit}

        if scrape:
            kwargs["scrape_options"] = {
                "formats": ["markdown"],
                "timeout": 25,
            }

        if include_domains:
            kwargs["include_domains"] = include_domains

        self._total_calls += 1
        if not scrape:
            self._discovery_calls += 1

        call_record = {
            "number": self._total_calls,
            "query": clean_query,
            "scrape": scrape,
            "limit": result_limit,
            "domains": include_domains or [],
            "status": "started",
            "results": 0,
        }
        self._call_history.append(call_record)

        try:
            search = _olostep_client.searches.create(**kwargs)
        except Olostep_BaseError as error:
            call_record["status"] = "error"
            call_record["error"] = f"{type(error).__name__}: {error}"
            return (
                f"Olostep search error: {type(error).__name__}: {error}\n"
                f"Budget used: {self._total_calls}/6 total calls; "
                f"{self._discovery_calls}/3 discovery calls."
            )

        if not search.links:
            call_record["status"] = "no_results"
            return (
                f"No results found for query: {clean_query!r}\n"
                f"Budget used: {self._total_calls}/6 total calls; "
                f"{self._discovery_calls}/3 discovery calls."
            )

        call_record["status"] = "success"
        call_record["results"] = len(search.links)

        chunks = [
            f"## Web results for: {clean_query!r}",
            f"Page scraping: {'enabled' if scrape else 'disabled'}",
            (
                f"Budget used: {self._total_calls}/6 total calls; "
                f"{self._discovery_calls}/3 discovery calls."
            ),
        ]

        for index, link in enumerate(search.links, 1):
            url = link.get("url", "")
            title = link.get("title") or url or "Untitled result"
            description = link.get("description") or "No description provided."

            chunks.append(
                f"### {index}. {title}\n"
                f"URL: {url}\n"
                f"Summary: {description}"
            )

            markdown = link.get("markdown_content")
            if scrape and markdown:
                body = markdown.strip()
                if len(body) > self.max_chars_per_page:
                    body = body[: self.max_chars_per_page] + "\n...[truncated]"
                chunks.append(f"Page content:\n{body}")

        return "\n\n".join(chunks)


olostep_search_tool = OlostepSearchTool()
print(f"Tool ready: {olostep_search_tool.name}")


Olostep API key loaded.
Tool ready: olostep_web_search


## 6. Create the agents

Define three worker agents and one manager agent. The manager is configured separately on the hierarchical crew and is not included in the worker list.


In [5]:
from crewai import Agent

researcher = Agent(
    role="Senior Research Analyst",
    goal=(
        "Find accurate, current, and well-sourced information about {topic}. "
        "Produce concrete, verifiable points for the writer."
    ),
    backstory=(
        "You are a meticulous research analyst. You decide whether each web "
        "search needs page scraping. Use scrape=False when titles, URLs, and "
        "summaries are enough. Use scrape=True when you need full-page evidence "
        "to verify a claim or understand a source in detail. Cite source URLs."
    ),
    llm=gemini_llm,
    tools=[olostep_search_tool],
    verbose=True,
    allow_delegation=False,
)


In [6]:
writer = Agent(
    role="Blog Post Writer",
    goal=(
        "Turn the research into a complete, publication-ready educational "
        "article on {topic}. Include a strong title, an unlabeled italic "
        "summary, TL;DR, detailed body sections, conclusion, and FAQs. Add a "
        "table only when it makes a comparison or decision easier to understand."
    ),
    backstory=(
        "You are a seasoned technology educator and writer. You create "
        "practical, approachable articles similar in structure to high-quality "
        "DataCamp content, without copying its wording or branding. You use "
        "clear headings, short paragraphs, concrete examples, and sourced facts."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False,
)


In [7]:
reviewer = Agent(
    role="Senior Editor",
    goal=(
        "Review the draft for factual accuracy, clarity, structure, and "
        "completeness. Ensure it follows the required article order and return "
        "a polished, publication-ready version."
    ),
    backstory=(
        "You are a detail-oriented senior editor for an educational technology "
        "publication. You verify claims, improve flow, remove repetition, and "
        "ensure the title is followed by an unlabeled italic summary and TL;DR. "
        "You keep a table only when it adds real value, remove any Key Takeaways "
        "section, and ensure Conclusion appears immediately before FAQs."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=False,
)


In [8]:
manager = Agent(
    role="Content Coordinator",
    goal=(
        "Coordinate research, writing, and review for {topic}. Delegate each "
        "task to the correct specialist and ensure the reviewed article is the "
        "final result."
    ),
    backstory=(
        "You are an experienced editorial lead. You keep the workflow simple: "
        "research first, writing second, and editorial review last."
    ),
    llm=gemini_llm,
    verbose=True,
    allow_delegation=True,
)


## 7. Define tasks

Create unassigned tasks for the manager to delegate. In a hierarchical process, the manager selects the most suitable worker and validates the result of each task.


In [9]:
from crewai import Task

research_task = Task(
    description=(
        "Delegate to the Senior Research Analyst. Research '{topic}' using at "
        "most 6 web-tool calls. Use no more than 3 calls with scrape=False, "
        "then use scrape=True only when detailed evidence is needed. Collect "
        "5-7 useful findings, source URLs, and common reader questions."
    ),
    expected_output=(
        "Clear markdown research notes with 5-7 sourced findings and 3-5 "
        "potential FAQ questions."
    ),
)

write_task = Task(
    description=(
        "Delegate to the Blog Post Writer. Write one complete educational "
        "markdown article on '{topic}', approximately 1,000-1,500 words. Start "
        "with one H1 title and an unlabeled italic summary. Then include TL;DR, "
        "an introduction, useful body sections, an optional table only when it "
        "improves understanding, a Conclusion, and FAQs. Use source links."
    ),
    expected_output=(
        "One continuous publication-ready markdown article with title, italic "
        "summary, TL;DR, body, Conclusion, and 3-5 FAQs."
    ),
    context=[research_task],
)

review_task = Task(
    description=(
        "Delegate to the Senior Editor. Review the article for accuracy, "
        "clarity, structure, and source support. Return only the corrected full "
        "article. Keep the same continuous article format and ensure FAQs are "
        "the final section."
    ),
    expected_output=(
        "Only the final reviewed markdown article, beginning with its H1 title "
        "and ending with the final FAQ answer."
    ),
    context=[research_task, write_task],
)

tasks = [research_task, write_task, review_task]

print(f"Defined {len(tasks)} manager-delegated tasks.")


Defined 3 manager-delegated tasks.


In [10]:
from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer, reviewer],
    tasks=tasks,
    manager_agent=manager,
    process=Process.hierarchical,
    verbose=True,
    memory=False,
)

print("Hierarchical crew assembled.")
print(f"  Workers : {[a.role for a in crew.agents]}")
print(f"  Manager : {crew.manager_agent.role}")
print(f"  Tasks   : {len(crew.tasks)}")
print(f"  Process : {crew.process}")


Hierarchical crew assembled.
  Workers : ['Senior Research Analyst', 'Blog Post Writer', 'Senior Editor']
  Manager : Content Coordinator
  Tasks   : 3
  Process : Process.hierarchical


## 9. Run the swarm

Provide a topic and start the hierarchical crew with `kickoff_async()`. The manager delegates tasks to the worker agents and validates their responses.


> Jupyter already runs an event loop. The next cell applies `nest_asyncio` so CrewAI can run in the notebook.


In [11]:
try:
    import nest_asyncio

    nest_asyncio.apply()

    print("nest_asyncio applied: CrewAI can now run inside this kernel.")

except ModuleNotFoundError:
    print("nest_asyncio not installed. Run:  pip install nest-asyncio")


nest_asyncio applied: CrewAI can now run inside this kernel.


In [12]:
from datetime import datetime

topic = "Learning Adaptive Thresholds for LLM Tool Use: When to Rely on Memory and When to Call a Tool"


async def run_crew(topic: str):
    """Run the CrewAI crew asynchronously."""
    print(f"Starting crew at {datetime.now().isoformat(timespec='seconds')}")
    print(f"Topic: {topic!r}\n")

    result = await crew.kickoff_async(inputs={"topic": topic})

    print(f"\nFinished at {datetime.now().isoformat(timespec='seconds')}")
    print(f"Result type: {type(result).__name__}")
    return result


result = await run_crew(topic)

Starting crew at 2026-06-05T13:54:07
Topic: 'Learning Adaptive Thresholds for LLM Tool Use: When to Rely on Memory and When to Call a Tool'



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 84eca850-ac80-454c-b99a-5b239b9d14c3                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Delegate to the Senior Research Analyst. Research 'Learning Adaptive Thresholds for LLM Tool Use: When   │
│  to Rely on Memory and When to Call a Tool' using at most 6 web-tool calls. Use no more than 3 calls with       │
│  scrape=False, then use scrape=True only when detailed evidence is needed. Collect 5-7 useful findings, source  │
│  URLs, and common reader questions.                                                                             │
│  ID: 200f78cf-91bb-4a54-91b9-410b96ffc042                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Coordinator                                                                                     │
│                                                                                                                 │
│  Task: Delegate to the Senior Research Analyst. Research 'Learning Adaptive Thresholds for LLM Tool Use: When   │
│  to Rely on Memory and When to Call a Tool' using at most 6 web-tool calls. Use no more than 3 calls with       │
│  scrape=False, then use scrape=True only when detailed evidence is needed. Collect 5-7 useful findings, source  │
│  URLs, and common reader questions.                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': "We are preparing a comprehensive blog post on the paper/topic: 'Learning Adaptive           │
│  Thresholds for LLM Tool Use: When to Rely on Memory and When to Call a Tool'. This paper addresses how La...   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: # Learning Adaptive Thresholds for LLM Tool Use: When to Rely on Memory and When to Call a Tool        │
│                                                                                                                 │
│  *By extracting meta-cognitive confidence scores directly from an LLM’s internal activation states, developers  │
│  can deploy adaptive dual-threshold policies that dynamically decide when to rely on parametric memory and      │
│  when to query external tools—slashing latency, minimizing operational costs, and preventing tool-induced       │
│  errors.*                                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## TL;DR                                                                                                       │
│  * **The Problem:** Indiscriminate tool use in Large Language Model (LLM) agents increases latency, drives up   │
│  operational API costs, and risks introducing errors when external tools return noisy or irrelevant data.       │
│  * **The Solution:** The Meta-Cognition Trigger (MeCo) framework reads an LLM's internal hidden states during   │
│  the initial forward pass to calculate a scalar "meta-cognitive score" of its own confidence.                   │
│  * **The Policy:** A dual-threshold policy routes queries dynamically: high-confidence queries rely entirely    │
│  on parametric memory, low-confidence queries trigger immediate tool execution, and borderline queries utilize  │
│  dynamic calibration.                                                                                           │
│  * **The Benefit:** This framework is completely fine-tuning-free, introduces near-zero computational           │
│  overhead, and has been validated across major open-source model families including Llama, Mistral, and Qwen.   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  In the deployment of Large Language Models (LLMs), parametric memory—the billions of weights that allow        │
│  models to recall facts, write code, and summarize text—has physical limits. Knowledge cutoffs,                 │
│  hallucinations, and logic errors are persistent challenges.                                                    │
│                                                                                                                 │
│  To overcome these constraints, developers built tool-use architectures, including Retrieval-Augmented          │
│  Generation (RAG), search APIs, and code interpreters. However, the industry has rapidly swung to the opposite  │
│  extreme: **indiscriminate tool use**. In many agentic 

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Coordinator                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Learning Adaptive Thresholds for LLM Tool Use: When to Rely on Memory and When to Call a Tool                │
│                                                                                                                 │
│  *By extracting meta-cognitive confidence scores directly from an LLM’s internal activation states, developers  │
│  can deploy adaptive dual-threshold policies that dynamically decide when to rely on parametric memory and      │
│  when to query external tools—slashing latency, minimizing operational costs, and preventing tool-induced       │
│  errors.*                                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## TL;DR                                                                                                       │
│  * **The Problem:** Indiscriminate tool use in Large Language Model (LLM) agents increases latency, drives up   │
│  operational API costs, and risks introducing errors when external tools return noisy or irrelevant data.       │
│  * **The Solution:** The Meta-Cognition Trigger (MeCo) framework reads an LLM's internal hidden states during   │
│  the initial forward pass to calculate a scalar "meta-cognitive score" of its own confidence.                   │
│  * **The Policy:** A dual-threshold policy routes queries dynamically: high-confidence queries rely entirely    │
│  on parametric memory, low-confidence queries trigger immediate tool execution, and borderline queries utilize  │
│  dynamic calibration.                                                                                           │
│  * **The Benefit:** This framework is completely fine-tuning-free, introduces near-zero computational           │
│  overhead, and has been validated across major open-source model families including Llama, Mistral, and Qwen.   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Introduction                                                                                                │
│                                                                                                                 │
│  In the deployment of Large Language Models (LLMs), parametric memory—the billions of weights that allow        │
│  models to recall facts, write code, and summarize text—has physical limits. Knowledge cutoffs,                 │
│  hallucinations, and logic errors are persistent challenges.                                                    │
│                                                                                                                 │
│  To overcome these constraints, developers built tool-use architectures, including Retrieval-Augmented          │
│  Generation (RAG), search APIs, and code interpreters. 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Delegate to the Senior Editor. Review the article for accuracy, clarity, structure, and source support.  │
│  Return only the corrected full article. Keep the same continuous article format and ensure FAQs are the final  │
│  section.                                                                                                       │
│  Agent: Content Coordinator                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Finished at 2026-06-05T13:59:29
Result type: CrewOutput


## 10. View the result

Display the complete article with its title, unlabeled italic summary, TL;DR, body, optional table, conclusion, and FAQs.


In [13]:
from IPython.display import Markdown, display

raw = result.raw

if "Blog Post" in raw:
    raw = raw.split("Blog Post", 1)[1].strip()

display(Markdown(raw))

# Learning Adaptive Thresholds for LLM Tool Use: When to Rely on Memory and When to Call a Tool

*By extracting meta-cognitive confidence scores directly from an LLM’s internal activation states, developers can deploy adaptive dual-threshold policies that dynamically decide when to rely on parametric memory and when to query external tools—slashing latency, minimizing operational costs, and preventing tool-induced errors.*

---

## TL;DR
* **The Problem:** Indiscriminate tool use in Large Language Model (LLM) agents increases latency, drives up operational API costs, and risks introducing errors when external tools return noisy or irrelevant data.
* **The Solution:** The Meta-Cognition Trigger (MeCo) framework reads an LLM's internal hidden states during the initial forward pass to calculate a scalar "meta-cognitive score" of its own confidence.
* **The Policy:** A dual-threshold policy routes queries dynamically: high-confidence queries rely entirely on parametric memory, low-confidence queries trigger immediate tool execution, and borderline queries utilize dynamic calibration.
* **The Benefit:** This framework is completely fine-tuning-free, introduces near-zero computational overhead, and has been validated across major open-source model families including Llama, Mistral, and Qwen.

---

## Introduction

In the deployment of Large Language Models (LLMs), parametric memory—the billions of weights that allow models to recall facts, write code, and summarize text—has physical limits. Knowledge cutoffs, hallucinations, and logic errors are persistent challenges. 

To overcome these constraints, developers built tool-use architectures, including Retrieval-Augmented Generation (RAG), search APIs, and code interpreters. However, the industry has rapidly swung to the opposite extreme: **indiscriminate tool use**. In many agentic workflows, an LLM calls an external search engine or database for *every single query*, regardless of whether it already possesses the correct answer internally. 

A smarter path forward lies in teaching LLMs to evaluate their own confidence in real time. Drawing on recent research, notably the **MeCo (Adaptive Tool Use in Large Language Models with Meta-Cognition Trigger)** framework submitted to ICLR 2025, we can analyze how models evaluate their own capability. By extracting internal activation patterns and applying adaptive dual thresholds, developers can build agents that dynamically balance internal memory and external tool execution.

---

## The Core Bottleneck: The Cost of Indiscriminate Tool Use

Integrating external APIs into LLM pipelines is powerful, but blind invocation introduces three major bottlenecks:

```
                  ┌────────────────────────┐
                  │       User Query       │
                  └───────────┬────────────┘
                              │
                    [ Naive Tool-Routing ]
                              │
             ┌────────────────┴────────────────┐
             ▼                                 ▼
    [ Call External API ]             [ Rely on Parametric Memory ]
     - High Latency (Seconds)          - Low Latency (Milliseconds)
     - High Cost ($$$)                 - Zero Extra Cost ($0)
     - Risk of Noisy Data/Errors       - Risk of Hallucination (if uncertain)
```

### 1. High Latency
Calling an external API—whether a search engine, SQL database, or code interpreter—adds a significant latency penalty. A query that could be resolved in milliseconds via local model inference can take several seconds due to network round-trips and external processing.

### 2. Escalating Operational Costs
For enterprise applications processing millions of tokens, third-party API fees accumulate rapidly. If an LLM can correctly answer a query using its pre-existing parameters, routing that query to an external service is an unnecessary operational expense.

### 3. Tool-Induced Error Propagation
External tools do not always return clean, helpful data. Search engines surface SEO-optimized noise, and databases retrieve irrelevant rows. When this irrelevant information is injected back into the LLM’s context window, it can corrupt the model's reasoning, causing it to fail on questions it could have answered correctly using its own internal knowledge.

---

## What is Meta-Cognition in LLMs?

To prevent indiscriminate tool use, an agent needs an intelligent gatekeeper. Traditional setups rely on **prompt-based routing**, where developers ask the model in natural language: *"Do you know the answer to this question, or do you need to search?"* 

This approach is fundamentally flawed. It requires generating extra text tokens, which increases both cost and latency. Furthermore, LLMs are prone to overconfidence and sycophancy when self-evaluating through natural language prompting.

The alternative introduced by the MeCo framework is **meta-cognition**—an agent's ability to monitor and evaluate its own internal cognitive states. In LLMs, this is achieved by examining the model's internal neural activity during the very first forward pass of a query, bypassing the need for text-based self-evaluation.

---

## Probing the Latent Representation Space

Measuring an LLM's confidence without generating text relies on **representation engineering** within the model's latent representation space.

When an LLM processes an input prompt, the tokens pass through dozens of transformer layers. Each layer generates high-dimensional vectors (hidden activations). High-level cognitive phenomena—such as certainty, uncertainty, and semantic conflict—are encoded as distinct activation patterns within these intermediate layers.

```
       Input Prompt ──► [ Layer 1 ] ──► [ Layer 16 ] ──► [ Layer 32 ] ──► Token Out
                                             │
                                   [ Representation Probe ]
                                             │
                                  Meta-Cognitive Score (S)
```

The MeCo framework uses a lightweight probe to scan these hidden states during inference:
* **The Signal:** As a query is encoded, the system extracts the activation vectors from designated intermediate layers of the neural network.
* **The Extraction:** By applying lightweight representation engineering, these vectors are mapped to a scalar **meta-cognitive score ($S$)** representing the model's confidence in its own parametric knowledge.
* **The Efficiency:** This probe runs during the prompt's initial forward pass. It requires no extra token generation and introduces virtually zero computational overhead.

---

## The Dual-Threshold Routing Policy

A single, static threshold (such as "if confidence is less than 0.5, call a tool") is too rigid to handle borderline cases safely. To solve this, advanced architectures implement a **dual-threshold policy** featuring a lower threshold ($\theta_{\text{low}}$) and an upper threshold ($\theta_{\text{high}}$).

```
                  [ Meta-Cognitive Score (S) ]
                               │
         ┌─────────────────────┼─────────────────────┐
         ▼                     ▼                     ▼
   S < θ_low             θ_low ≤ S < θ_high          S ≥ θ_high
 [Low Confidence]       [Uncertainty Zone]     [High Confidence]
         │                     │                     │
         ▼                     ▼                     ▼
   Trigger Tool        Calibrate / Ask Tool    Parametric Memory Only
(e.g., Web Search)       (Dynamic Route)         (Zero Tool Calls)
```

### 1. High-Confidence Zone ($S \ge \theta_{\text{high}}$)
If the meta-cognitive score meets or exceeds the upper threshold, the model has high certainty. The system bypasses external tools entirely and answers immediately from its parametric memory, minimizing latency and cost.

### 2. Low-Confidence Zone ($S < \theta_{\text{low}}$)
If the score falls below the lower threshold, the system is highly uncertain. The LLM immediately triggers the external tool (e.g., search engine, calculator, or database retriever) rather than risking a hallucination.

### 3. Weak/Uncertainty Zone ($\theta_{\text{low}} \le S < \theta_{\text{high}}$)
This represents the borderline zone where self-assessment is uncertain. Here, the system can apply dynamic or task-specific calibration—such as falling back to a lightweight verification prompt or choosing a lower-cost retrieval tool—to determine the safest, most efficient path.

### Adaptive Calibration
These thresholds are highly adaptable because they can be calibrated on a small validation dataset, allowing developers to set custom performance trade-offs:
* **High-Stakes Mode (e.g., Medical or Financial Agents):** Push both thresholds higher. This forces the model to verify facts via external databases at the slightest hint of uncertainty.
* **Cost/Latency Optimization Mode (e.g., High-Volume Support Chatbots):** Lower the thresholds to prioritize fast, local parametric responses, invoking tools only when the model is highly uncertain.

---

## Why Adaptive Routing is a Game-Changer

To understand the benefits of this approach, we can compare adaptive threshold routing against traditional methods:

| Criteria | Indiscriminate Tool Use | Prompt-Based Routing | Adaptive / MeCo Threshold Routing |
| :--- | :--- | :--- | :--- |
| **Average Latency** | High (Always waits for network APIs) | High (Requires generating self-eval tokens) | **Low** (Bypasses tools for known queries) |
| **Operational Cost** | High (Paid API charges per query) | Medium (Extra token processing fees) | **Low** (Zero API fees for known queries) |
| **Risk of Error Propagation** | High (Noisy tools confuse the model) | Low/Medium (Depends on prompt quality) | **Low** (Filters out noisy external data) |
| **Fine-Tuning Required** | None | None (Uses system prompts) | **None** (Uses representation engineering) |
| **Decision Overhead** | Zero (Fixed path) | High (Runs full inference cycles) | **Insignificant** (Reads hidden layer activations) |
| **Adaptability** | None (Static) | Low (Hard to adjust prompt behavior) | **High** (Adjustable thresholds via validation calibration) |

---

## Fine-Tuning-Free and Cost-Efficient Integration

A common barrier to deploying advanced LLM optimizations is the need for fine-tuning, which requires specialized hardware, clean training datasets, and significant engineering hours. 

The MeCo framework is entirely **fine-tuning-free**. It does not modify the base LLM's weights. Because it operates directly on raw layer activations, developers can integrate it as a lightweight middleware observer. 

This brings several practical advantages:
* **Plug-and-Play Compatibility:** It can be added to off-the-shelf, open-source weights without breaking existing model behaviors.
* **Portability:** Upgrading a base model (e.g., from Llama-3-8B to Llama-3-70B) does not require retraining; developers only need to run a quick calibration step to map the new model's hidden layer activations.
* **Zero-Overhead Integration:** Computing the meta-cognitive score from hidden activations during the standard forward pass takes milliseconds, introducing virtually zero computational overhead.

---

## Robust Experimental Validation

The adaptive threshold methodology has been validated across multiple open-source model families and complex benchmarks:

1. **Model Families Evaluated:** Researchers verified the framework across popular open-source LLMs, including **Llama, Mistral, and Qwen**.
2. **Diverse Task Domains:** Evaluations covered complex, knowledge-intensive, and reasoning tasks, testing models against various external tools like search APIs, calculators, and code interpreters.
3. **Performance Gains:** 
   * **Accuracy Improvements:** By pruning unnecessary, noisy external tool outputs, the adaptive framework consistently outperformed "always-use-tools" baselines and prompt-based routing strategies.
   * **Reduced API Overhead:** The framework significantly reduced redundant tool calls while maintaining or improving final task accuracy, leading to substantial savings in API costs and execution latency.

---

## Conclusion

The future of LLM agents lies in balanced execution. Relying entirely on frozen parametric memory leads to hallucinations, while treating every query as an API search emergency introduces latency and high operational costs. 

By analyzing the latent representation space of LLMs, frameworks like MeCo capture the model's internal confidence. Applying a dual-threshold routing policy enables the best of both worlds: lightning-fast, zero-cost parametric responses for what the model already knows, and robust, automated tool use for what it does not.

---

## Frequently Asked Questions (FAQs)

### Q1: Why is indiscriminate tool invocation a problem in LLM systems?
When LLM agents call tools (like web search engines, calculators, or databases) for every user request, they encounter three primary bottlenecks:
* **High Latency:** Waiting for external network API requests can turn milliseconds of processing into several seconds of user waiting time.
* **Elevated Costs:** Continuous API calls scale up operational and token expenses rapidly.
* **Error Propagation:** If an external tool retrieves noisy, outdated, or irrelevant information, the LLM can become confused, leading to failures on queries the model could have answered correctly using its internal memory.

### Q2: How does the system compute a "meta-cognitive score" without slow prompting?
Instead of asking the LLM to write out whether it knows an answer (which requires slow and expensive token generation), the system extracts the score directly from the model's internal **representation space**. By reading hidden states/activations at intermediate layers during the initial forward pass of the query, it detects latent representation patterns of confidence versus uncertainty, producing a fast scalar score with virtually zero computational overhead.

### Q3: How are the thresholds "adaptive"? Can I adjust them for my specific use case?
The thresholds are adaptive because they are calibrated on a small validation set of sample queries. Developers can adjust the upper ($\theta_{\text{high}}$) and lower ($\theta_{\text{low}}$) boundaries to match specific performance trade-offs. For high-stakes applications (like medical or financial advice), you can raise the thresholds so the model relies on tools at the slightest hint of uncertainty. For latency-critical applications, you can lower them to prioritize fast parametric answers.

### Q4: Does implementing an adaptive threshold system require fine-tuning or retraining the LLM?
No. Frameworks like MeCo are entirely **fine-tuning-free**. They leverage the existing, pre-trained weights of the base LLM. The system only reads internal activations and applies a calibrated mathematical formula to compute the score, making it a highly lightweight, plug-and-play middleware that is compatible with off-the-shelf open-source models.

### Q5: How does this representation-based approach differ from traditional semantic search routing?
Semantic routing models choose a tool based entirely on the semantic meaning of the input text. However, they cannot evaluate if the LLM's parametric memory already contains the correct answer. For example, a semantic router might see the question "What is the capital of France?" and route it to a search engine because it matches the category of a geography query. An adaptive, representation-based threshold system looks at the model's internal confidence, realizes it knows the answer with high certainty, and skips the external tool call entirely.

### Q6: Can this approach handle complex multi-step tool calls?
Yes. While the primary decision occurs during the initial query evaluation, the framework can be applied iteratively. At each step of an agentic chain-of-thought, the model's internal activations can be monitored to determine if the next reasoning step is highly confident or requires calling another specialized tool (e.g., calling a calculator mid-way through a math reasoning problem).

In [14]:
from collections import Counter


def role_name(value):
    """Return a readable role/name for CrewAI agent values."""
    if value is None:
        return "Not reported"
    if isinstance(value, str):
        return value
    return getattr(value, "role", None) or getattr(value, "name", None) or str(value)


print("=" * 70)
print("SWARM EXECUTION TRACE")
print("=" * 70)

# Hierarchical crew structure
print("\nCOORDINATION")
print(f"Process : {crew.process}")
print(f"Manager : {role_name(crew.manager_agent)}")
print(f"Workers : {', '.join(role_name(agent) for agent in crew.agents)}")
print(
    "Flow    : Manager receives each task, delegates specialist work, "
    "validates the response, and passes approved context to the next task."
)

# Web-tool activity recorded directly by the custom Olostep tool
history = list(getattr(olostep_search_tool, "_call_history", []))
discovery_calls = sum(not item["scrape"] for item in history)
scrape_calls = sum(item["scrape"] for item in history)
successful_calls = sum(item["status"] == "success" for item in history)

print("\nWEB TOOL USAGE")
print(f"Total external calls : {len(history)}/6")
print(f"Without scraping     : {discovery_calls}/3")
print(f"With scraping        : {scrape_calls}")
print(f"Successful calls     : {successful_calls}")

if history:
    for item in history:
        mode = "scrape" if item["scrape"] else "search only"
        domains = ", ".join(item["domains"]) if item["domains"] else "all domains"
        print(
            f"  {item['number']}. {mode}; status={item['status']}; "
            f"results={item['results']}; scope={domains}"
        )
        print(f"     Query: {item['query']}")
else:
    print("  No Olostep calls were recorded.")

# Task and delegation details
print("\nTASK-BY-TASK TRACE")
agent_activity = Counter()

for index, task_out in enumerate(result.tasks_output, start=1):
    task = crew.tasks[index - 1] if index <= len(crew.tasks) else None
    reported_agent = role_name(getattr(task_out, "agent", None))
    processed_by = sorted(getattr(task, "processed_by_agents", set()) or [])
    participants = processed_by or [reported_agent]

    for participant in participants:
        agent_activity[role_name(participant)] += 1

    delegations = getattr(task, "delegations", 0) if task else 0
    used_tools = getattr(task, "used_tools", 0) if task else 0
    tool_errors = getattr(task, "tools_errors", 0) if task else 0
    start_time = getattr(task, "start_time", None) if task else None
    end_time = getattr(task, "end_time", None) if task else None
    duration = end_time - start_time if start_time and end_time else None

    print(f"\n--- Task {index} ---")
    print(f"Reported owner : {reported_agent}")
    print(f"Processed by   : {', '.join(map(role_name, participants))}")
    print(f"Delegations    : {delegations}")
    print(f"Tool uses      : {used_tools}")
    print(f"Tool errors    : {tool_errors}")
    if duration is not None:
        print(f"Duration       : {duration}")

    preview = getattr(task_out, "raw", "").strip().replace("\n", " ")
    print(f"Output preview : {preview[:300]}{'...' if len(preview) > 300 else ''}")

print("\nAGENT ACTIVITY")
for agent, count in agent_activity.most_common():
    print(f"{agent}: participated in {count} task(s)")

if not agent_activity:
    print("CrewAI did not expose per-agent processing metadata for this run.")


SWARM EXECUTION TRACE

COORDINATION
Process : Process.hierarchical
Manager : Content Coordinator
Workers : Senior Research Analyst, Blog Post Writer, Senior Editor
Flow    : Manager receives each task, delegates specialist work, validates the response, and passes approved context to the next task.

WEB TOOL USAGE
Total external calls : 6/6
Without scraping     : 3/3
With scraping        : 3
Successful calls     : 6
  1. search only; status=success; results=3; scope=all domains
     Query: Learning Adaptive Thresholds for LLM Tool Use: When to Rely on Memory and When to Call a Tool
  2. search only; status=success; results=3; scope=all domains
     Query: "Learning Adaptive Thresholds for LLM Tool Use"
  3. search only; status=success; results=3; scope=all domains
     Query: "Adaptive Thresholds" LLM "Tool Use" memory OR tool
  4. scrape; status=success; results=3; scope=all domains
     Query: "When to Rely on Memory and When to Call a Tool"
  5. scrape; status=success; results=3; sco